# K03_01 – Metriken bei Klassifikationsmodellen – Studentenversion

Diese Fassung ist für die **aktive Mitarbeit im Kurs** gedacht.

In diesem Notebook bewerten wir ein **Klassifikationsmodell**.  
Wir konzentrieren uns auf die wichtigsten Kennzahlen:
- Accuracy
- Konfusionsmatrix
- Precision
- Recall
- F1-Score

## Lernziele
Nach diesem Notebook können Sie:
- die Güte eines Klassifikators auf Testdaten bewerten
- eine Konfusionsmatrix lesen
- Precision und Recall fachlich einordnen
- erklären, warum Accuracy allein nicht immer ausreicht

## 1. Datensatz und Train/Test-Split

Wir verwenden den bekannten Datensatz `breast_cancer`.  
Das ist ein **binäres Klassifikationsproblem**.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X = data.data
y = data.target

print("Shape von X:", X.shape)
print("Shape von y:", y.shape)
print("Klassen:", data.target_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


## 2. Modell trainieren

Wir nutzen eine einfache Pipeline aus:
- `StandardScaler`
- `LogisticRegression`


In [ ]:
clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42)
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)


## 3. Accuracy

Die Accuracy misst den Anteil korrekt klassifizierter Beispiele.


In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.3f}")


## Mini-Übung 1
Interpretieren Sie den Wert:
- Ist die Accuracy eher hoch oder eher niedrig?
- Reicht diese Zahl allein schon aus, um das Modell vollständig zu bewerten?


## 4. Konfusionsmatrix

Die Konfusionsmatrix zeigt, **welche Arten von Fehlern** das Modell macht.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=data.target_names)
disp.plot()
plt.show()


### Erinnerung
- **TP**: positiv und korrekt erkannt
- **TN**: negativ und korrekt erkannt
- **FP**: fälschlich positiv
- **FN**: fälschlich negativ


## Mini-Übung 2
Schauen Sie in die Konfusionsmatrix:
1. Wie viele Fehlklassifikationen gibt es insgesamt?
2. Welche Fehlerart wäre in der Medizin oft kritischer: FP oder FN?


In [ ]:
fehler = (cm[0,1] + cm[1,0])
print("Anzahl Fehlklassifikationen:", fehler)


## 5. Precision, Recall und F1-Score


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-Score:  {f1:.3f}")

print("\nKlassifikationsbericht:")
print(classification_report(y_test, y_pred, target_names=data.target_names))


### Fachliche Deutung
- **Precision**: Wie viele als positiv vorhergesagte Fälle sind wirklich positiv?
- **Recall**: Wie viele der tatsächlich positiven Fälle wurden gefunden?
- **F1-Score**: harmonisches Mittel aus Precision und Recall


## 6. Warum Accuracy allein täuschen kann

Wir konstruieren jetzt ein unausgewogenes Beispiel und vergleichen ein triviales Modell.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier

X_imb, y_imb = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=4,
    n_redundant=0,
    weights=[0.95, 0.05],
    random_state=42
)

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb
)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_i, y_train_i)
y_pred_dummy = dummy.predict(X_test_i)

print("Accuracy des Dummy-Modells:", round(accuracy_score(y_test_i, y_pred_dummy), 3))
print("Recall des Dummy-Modells:", round(recall_score(y_test_i, y_pred_dummy, zero_division=0), 3))
print("Precision des Dummy-Modells:", round(precision_score(y_test_i, y_pred_dummy, zero_division=0), 3))


## Mini-Übung 3
Warum wirkt die Accuracy des Dummy-Modells zunächst gut, obwohl das Modell fachlich unbrauchbar ist?


## 7. Zusammenfassung

Für Klassifikationsmodelle gilt:
- Accuracy ist ein guter erster Überblick
- die Konfusionsmatrix zeigt die Fehlerarten
- Precision und Recall beantworten unterschiedliche fachliche Fragen
- bei unausgewogenen Klassen ist Accuracy allein oft zu wenig
